In [ ]:
import numpy as np 
import pandas as pd 
import pyfolio as pf
from pyfolio import timeseries
import matplotlib.pylab as plt
from backtest import  backtest_strat, baseline_strat
%matplotlib inline
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)


In [ ]:
data = pd.read_csv("datasets/stock20_Basic.csv")

In [ ]:
list_stock = data.tic.unique().tolist()

In [ ]:
list_stock

In [ ]:
len(list_stock)

In [ ]:
data_test = pd.read_csv("../datasets/S&P500_ClosePrice.csv")

In [ ]:
len(list(data_test.columns))

In [ ]:
data_test.columns.isin(list_stock)

In [ ]:
list_stock.append("date")

In [ ]:
data_test = data_test[list_stock]

In [ ]:
data_test = data_test.set_index("date")

In [ ]:
data_test = data_test[(data_test.index >= "2023-01-01" ) & (data_test.index <"2024-01-01")]

In [ ]:
# Define a Function for Displaying the Cleaned Weights
def show_clean_p(port_df):
    p1_show_1 = (port_df.transpose()[0]).map(lambda x: "{:.3%}".format(x)).to_frame().transpose()
    return display(HTML(p1_show_1.to_html()))

In [ ]:
ticker_list = list(data_test.columns) # Get List of all ticker symbols
n_assets = len(ticker_list) # Number of assets

uniform_weights = np.ones((n_assets))/n_assets

uniform_weights_port = pd.DataFrame([uniform_weights], columns = ticker_list)

# print("\nuniform weights portfolio:\n")
# show_clean_p(uniform_weights_port)

In [ ]:
uniform_weights_port

In [ ]:
uw_weights = uniform_weights_port.values.flatten()
returns_test = data_test.pct_change()
uw_test_returns = returns_test.dot(uw_weights)
uw_test_cum_returns = (1 + uw_test_returns).cumprod()
uw_test_cum_returns.name = "2: Uniform weights"

In [ ]:
date_strings = uw_test_cum_returns.index.astype(str)
# 去除时间部分，只保留日期
date_strings = [date.split(' ')[0] for date in date_strings]
# 更新DataFrame的索引
uw_test_cum_returns.index = date_strings
# 查看更新后的索引
uw_test_cum_returns.index

In [ ]:
CNNandA3C_test1 = pd.read_csv("result/04_a3c_test_daily_return_CNN.csv")
CNNandA3C_test2 = pd.read_csv("result/04_a3c_test_daily_return_CNN1.csv")
CNNandA3C_test3 = pd.read_csv("result/04_a3c_test_daily_return_CNN2.csv")
CNNandA3C_test4 = pd.read_csv("result/04_a3c_test_daily_return_CNN3.csv")
CNNandA3C_test5 = pd.read_csv("result/04_a3c_test_daily_return_CNN4.csv")
CNNandA3C_test = pd.DataFrame()
CNNandA3C_test["daily_return"] = (CNNandA3C_test1["daily_return"] + CNNandA3C_test2["daily_return"] + CNNandA3C_test3["daily_return"] + CNNandA3C_test4["daily_return"] + CNNandA3C_test5["daily_return"]) / 5.0
CNNandA3C_test["date"] = CNNandA3C_test1["date"]

In [ ]:
a3c_test_cum_returns_conv = (1 + CNNandA3C_test["daily_return"]).cumprod()
a3c_test_cum_returns_conv.name = "1: CRL"

In [ ]:
fig, ax = plt.subplots(figsize=(12,6))
a3c_test_cum_returns_conv.plot(ax=ax, color='red', alpha=.6)
uw_test_cum_returns.plot(ax=ax, color="blue", alpha=.6)

plt.legend(loc="best");
plt.grid(True);
ax.set_ylabel("Daily return");
ax.set_xlabel("t");

In [ ]:
moth_A3C_CNN = pd.Series()
moth_A3C_CNN.name = "1: CRL(Our)"
moth_uw = pd.Series()
moth_uw.name = "2: Uniform_weights"
new_index = []
moth = 1
for i in range(0,len(uw_test_cum_returns),5):
    new_index.append(uw_test_cum_returns.index[i])
    moth_A3C_CNN[moth] = a3c_test_cum_returns_conv[i]
    moth_uw[moth] = uw_test_cum_returns[i]
    moth += 1

new_index.append(uw_test_cum_returns.index[-1])
new_index
# moth_A3C_CNN
moth_A3C_CNN[moth+1] = a3c_test_cum_returns_conv[len(a3c_test_cum_returns_conv)-1]
moth_A3C_CNN.index = new_index

moth_uw[moth+1] = uw_test_cum_returns[len(uw_test_cum_returns)-1]
moth_uw.index = new_index

In [ ]:
moth_uw[0] = 1

In [ ]:
fig, ax = plt.subplots(figsize=(12,6))

moth_A3C_CNN.plot(ax=ax, color='red', alpha=.6)
moth_uw.plot(ax=ax, color='blue', alpha=.6)

plt.legend(loc="best");
plt.grid(True);
ax.set_ylabel("Weekly return");
ax.set_xlabel("t");
# plt.xticks(rotation=45)
ax.set_title("Backtest based on the data from 2023-01-01 to 2023-12-31", fontsize=14);

In [ ]:
# Define a Function for Getting the Portfolio Statistics

def portfolio_stats(portfolio_returns):
    # Pass the returns into a dataframe
    port_rets_df = pd.DataFrame(portfolio_returns)
    port_rets_df = port_rets_df.reset_index()
    port_rets_df.columns = ['date','daily_return']
    #Use the FinRL Library to get the Portfolio Returns
    #This makes use of the Pyfolio Library
    DRL_strat = backtest_strat(port_rets_df)
    perf_func = timeseries.perf_stats 
    perf_stats_all = perf_func( returns=DRL_strat, 
                                factor_returns=DRL_strat, 
                                positions=None, transactions=None, turnover_denom="AGB")
    perf_stats_all = pd.DataFrame( perf_stats_all)
    perf_stats_all.columns = ['Statistic']
    return perf_stats_all
# Get the Portfolio Statistics for all the portfolios
portfolios_returns_dict = {
                          'CRL(Ours)':CNNandA3C_test['daily_return'],
                          'UW': uw_test_returns
                          }

portfolios_stats = pd.DataFrame()
for i,j in portfolios_returns_dict.items():
    port_stats = portfolio_stats(j)
    portfolios_stats[i] = port_stats['Statistic']
rows_to_select =[1,3,4,6,8]
portfolios_stats_new = portfolios_stats.iloc[rows_to_select] * 100
portfolios_stats_new.T

